# 🎬 Aiducation TikTok - Render Video Từ GitHub Trực Tiếp Trên Google Colab

Repository: [https://github.com/RevenantKitana/VideoCreation](https://github.com/RevenantKitana/VideoCreation)

Sổ tay này tự động đồng bộ mã nguồn từ **GitHub**, nhận file kịch bản `script.json` và **xuất video giáo dục hoàn chỉnh** (`final.mp4` + `cover.png`) trên phần cứng miễn phí của Google Colab.

> **Lưu ý:** Tuân thủ 100% điều khoản Google Colab (không mở SSH, thực thi trực tiếp trên giao diện web).

### 1. Đồng Bộ Mã Nguồn Từ GitHub & Kết Nối Google Drive

In [ ]:
# @title 1. Tải và đồng bộ dự án từ GitHub (RevenantKitana/VideoCreation)
import os
from pathlib import Path
from google.colab import drive

# 1. Kết nối Google Drive để tự động lưu video đầu ra vĩnh viễn
drive.mount('/content/drive')

REPO_URL = "https://github.com/RevenantKitana/VideoCreation.git"
PROJECT_DIR = Path("/content/VideoCreation")

# 2. Clone hoặc cập nhật từ GitHub
if not PROJECT_DIR.exists():
    print(f"-> Đang tải mã nguồn từ GitHub: {REPO_URL}...")
    !git clone {REPO_URL} {PROJECT_DIR}
    print("-> Đã tải mã nguồn thành công!")
else:
    print(f"-> Đang cập nhật mã nguồn mới nhất từ GitHub (git pull)...")
    %cd {PROJECT_DIR}
    !git pull origin main

%cd {PROJECT_DIR}

# 3. Tự động liên kết mô hình aligner vi_ctc.onnx nếu có trên Google Drive
aligner_model = PROJECT_DIR / "models/aligner/vi_ctc.onnx"
if not aligner_model.exists():
    drive_model = Path("/content/drive/MyDrive/xuong-tiktok/models/aligner/vi_ctc.onnx")
    if drive_model.exists():
        print("-> Đang đồng bộ mô hình kiểm âm vi_ctc.onnx từ Google Drive...")
        !mkdir -p {PROJECT_DIR}/models/aligner
        !cp {drive_model} {aligner_model}
        print("-> Đã đồng bộ vi_ctc.onnx thành công!")

print(f"\n✅ Dự án đã sẵn sàng tại: {PROJECT_DIR}")

### 2. Cài Đặt Môi Trường Render Linux (Chỉ chạy 1 lần khi mở Colab)

In [ ]:
# @title 2. Cài đặt các thư viện hệ thống và công cụ Render
!apt-get update -qq && apt-get install -y -qq \
    libnss3 libgbm1 libasound2 libatk-bridge2.0-0 libgtk-3-0 libxshmfence1 > /dev/null

# Chạy thiết lập môi trường Linux (Pixi, Manim, Node/Remotion, Voice Models)
!bash setup/setup-linux.sh

### 3. Nhập Kịch Bản `script.json`
Dán nội dung JSON vào giữa `r'''` và `'''` bên dưới:

In [ ]:
# @title 3. Nhập kịch bản JSON (Dùng tiền tố r''' để tránh lỗi LaTeX escape)
import json
from pathlib import Path

SCRIPT_CONTENT = r'''{
  "id": "V-0002",
  "slug": "giai-phuong-trinh-bac-hai",
  "title": "Giải phương trình bậc hai",
  "subject": "TOÁN",
  "voice": "Minh Quân Pro",
  "format": "9:16",
  "captions": "on",
  "scenes": [
    {
      "heading": "Giải phương trình bậc 2",
      "subheading": "Toán 9",
      "steps": [
        {
          "say": "Giải nhanh phương trình x bình phương trừ năm x cộng sáu bằng không trong ba mươi giây.",
          "show": [
            { "id": "de-bai", "math": "x^2 - 5x + 6 = 0" }
          ]
        },
        {
          "say": "Chúng ta sử dụng phương pháp phân tích đa thức thành nhân tử.",
          "show": [
            { "id": "phuong-phap", "text": "Phương pháp: Phân tích thành nhân tử", "style": "deep" }
          ]
        }
      ]
    },
    {
      "heading": "Phân tích nhân tử",
      "subheading": "Tìm hai số",
      "steps": [
        {
          "say": "Tìm hai số có tích bằng sáu và có tổng bằng trừ năm.",
          "show": [
            { "id": "dieu-kien", "rich": "Tìm hai số: $\\text{tích} = 6$ và $\\text{tổng} = -5$" }
          ]
        },
        {
          "say": "Đó chính là trừ hai và trừ ba.",
          "show": [
            { "id": "hai-so", "math": "(-2) \\times (-3) = 6 \\quad \\text{và} \\quad (-2) + (-3) = -5", "style": "accent" }
          ]
        },
        {
          "say": "Phương trình được viết lại thành x trừ hai nhân x trừ ba bằng không.",
          "show": [
            { "id": "nhan-tu", "math": "(x - 2)(x - 3) = 0" }
          ]
        }
      ]
    },
    {
      "heading": "Tìm nghiệm",
      "steps": [
        {
          "say": "Tương đương x trừ hai bằng không hoặc x trừ ba bằng không.",
          "show": [
            { "id": "nghiem-1", "math": "\\begin{cases} x - 2 = 0 \\\\ x - 3 = 0 \\end{cases}" }
          ]
        },
        {
          "say": "Vậy phương trình có hai nghiệm phân biệt là x bằng hai và x bằng ba.",
          "show": [
            { "id": "nghiem-2", "math": "\\begin{cases} x = 2 \\\\ x = 3 \\end{cases}", "style": "accent" }
          ]
        }
      ]
    },
    {
      "heading": "Ghi nhớ",
      "steps": [
        {
          "say": "Phương pháp nhẩm nghiệm theo tích và tổng giúp giải nhanh phương trình bậc hai.",
          "show": [
            { "id": "takeaway", "text": "Nhẩm nghiệm: Tích = 6, Tổng = -5", "style": "accent" }
          ]
        }
      ]
    }
  ]
}'''

script_data = json.loads(SCRIPT_CONTENT)
VIDEO_ID = script_data.get('id', 'V-0002')
video_folder = PROJECT_DIR / f"videos/{VIDEO_ID}-{script_data.get('slug', 'lesson')}"
video_folder.mkdir(parents=True, exist_ok=True)

script_file = video_folder / "script.json"
script_file.write_text(json.dumps(script_data, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✅ Đã lưu kịch bản thành công: {script_file}")

### 4. Kiểm Tra & Xem Trước Ảnh Khung Hình (Preview)

In [ ]:
# @title 4. Xuất và xem ảnh Preview
from IPython.display import Image, display
from pathlib import Path

print("-> Đang kiểm tra kịch bản và tạo ảnh xem trước (Preview)...")
!chmod +x os
!./os preview {VIDEO_ID}

preview_img = video_folder / "preview.png"
if preview_img.exists():
    print(f"✅ Đã tạo ảnh preview thành công: {preview_img}")
    display(Image(filename=str(preview_img), width=800))
else:
    print("⚠️ Chưa tạo được preview.png, vui lòng kiểm tra thông báo lỗi bên trên.")

### 5. Xuất Video Hoàn Chỉnh (Final Render)
Tạo giọng đọc AI (VieNeu), phụ đề karaoke và ghép video Remotion Full HD. Video sẽ được tự động lưu vào Google Drive của bạn.

In [ ]:
# @title 5. Xuất video hoàn chỉnh, Lưu vào Drive và Phát trực tiếp
from IPython.display import HTML, display
import base64
from pathlib import Path

print("-> 1/2: Đang tạo giọng đọc AI (voice)...")
!./os voice {VIDEO_ID}

print("\n-> 2/2: Đang ghép video và xuất file hoàn chỉnh (render)... (khoảng 3-5 phút)")
!./os render {VIDEO_ID}

video_path = video_folder / "final.mp4"
cover_path = video_folder / "cover.png"

if video_path.exists():
    print(f"\n🎉 XUẤT VIDEO THÀNH CÔNG!")
    print(f"📁 File video: {video_path}")
    print(f"📁 File ảnh bìa: {cover_path}")
    
    # Tự động sao lưu video vào Google Drive
    drive_output_dir = Path("/content/drive/MyDrive/VideoCreation_Outputs")
    drive_output_dir.mkdir(parents=True, exist_ok=True)
    drive_video = drive_output_dir / f"{VIDEO_ID}_final.mp4"
    !cp "{video_path}" "{drive_video}"
    print(f"💾 Đã lưu bản sao vĩnh viễn vào Google Drive: {drive_video}")
    
    # Hiển thị video player phát trực tiếp trong Colab
    video_bytes = open(video_path, "rb").read()
    video_b64 = base64.b64encode(video_bytes).decode("utf-8")
    display(HTML(f'''
        <h3>Xem video thành phẩm:</h3>
        <video width="360" height="640" controls autoplay style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
            Trình duyệt không hỗ trợ phát video này.
        </video>
    '''))
else:
    print("⚠️ Không tìm thấy final.mp4, vui lòng kiểm tra lỗi ở trên.")